# AIoT-DA L6: Multiple Linear Regression (50 Startups Kaggle)
## 跨領域資料分析作業流程 (CRISP-DM) 完整實作與 ChatGPT 協作指引

---

### 📌 本單元學習重點
1. **ChatGPT 提示詞工程協作**：學習如何透過結構化 Prompt 逐步引導 ChatGPT 完成從需求分析、代碼生成、錯誤修正到特徵工程。
2. **CRISP-DM 方法論標準流程**：遵循跨領域資料探勘 6 大標準步驟（Business Understanding $\rightarrow$ Data Understanding $\rightarrow$ Data Preparation $\rightarrow$ Modeling $\rightarrow$ Evaluation $\rightarrow$ Deployment）。
3. **資料前處理核心觀念**：
   - 名義變數的 One-Hot Encoding（獨熱編碼）。
   - **虛擬變數陷阱 (Dummy Variable Trap)** 的成因與解決方法（`drop='first'`）。
   - `train_test_split` 的必要性（評估未見資料的泛化能力，防範過擬合）。
4. **多元線性迴歸五大基本假定**：Linearity, Homoscedasticity, Multivariate normality, Independence of error, Lack of multicollinearity。
5. **模型評估指標意涵**：深入解析 $R^2$、MAE、MSE、RMSE 的數學意義與商業價值。
6. **過擬合與欠擬合判斷**：透過訓練與測試學習曲線 (Learning Curve) 診斷模型狀態。
7. **特徵工程與前向選擇 (Forward Selection)**：從 1 到 5 個特徵逐步挑選最優子集，實證 **Top 2（研發 + 行銷）** 為最佳架構。
8. **模型部署與序列化**：使用 `pickle` 與 `gzip` 完成輕量化壓縮部署與生產環境推論。

## 💬 與 ChatGPT 協作對話紀錄 (Prompt Engineering Guide)

> **Step 1 Prompt (資料取得與基本探勘)**:
> *"solve 50 startup multiple linear regression problem in kaggle using LR, applying the CRISP-DM steps. I want to load 50 startup data from web using URL https://gist.githubusercontent.com/GaneshSparkz/b5662effbdae8746f7f7d8ed70c42b2d/raw/faf8b1a0d58e251f48a647d3881e7a960c3f0925/50_Startups.csv. Convert to pandas DataFrame and print summary."*

> **Step 2 Prompt (資料前處理與虛擬變數處理)**:
> *"Separate features (X) and target (y). Handle categorical variable State using OneHotEncoder with drop='first' to avoid dummy variable trap. Split dataset into train and test with test_size=0.2 and random_state=0. Explain why train_test_split is necessary."*

> **Step 3 & 4 Prompt (建模與評估指標)**:
> *"Train a LinearRegression model. Compute R2, MAE, MSE, and RMSE. Plot Actual vs Predicted values. Also plot a learning curve comparing training error vs test error to judge overfitting vs underfitting."*

> **Step 5 Prompt (前向特徵選擇 1 to 5 特徵)**:
> *"Implement forward selection for 1 to 5 features based on minimizing RMSE and maximizing R2. Display the results in a Markdown table format comparing Number of Features, Selected Features, RMSE, and R-squared."*

> **Step 6 Prompt (部署)**:
> *"Export the trained optimal model using pickle and gzip. Show how to reload and perform inference on a new startup."*

---
## 🏢 CRISP-DM Step 1: Business Understanding (商業理解)

- **商業目標**：創業投資機構（Venture Capital, VC）每年需審核大量新創團隊。本專案目標是建立一個客觀量化的預測模型，根據新創公司在各項領域的支出（研發 R&D、行政管理 Administration、行銷行銷 Marketing）以及所在地點（State: California, Florida, New York），精確預測其獲利能力（Profit）。
- **利害關係人**：創投合夥人、天使投資人、新創執行長、孵化器經理人。
- **成功準則**：模型在測試集上的決定係數 $R^2 > 0.85$，RMSE 小於預測利潤均值的 10%，且滿足迴歸假定。

---
## 📊 CRISP-DM Step 2: Data Understanding (資料理解)

本步驟載入資料集，檢驗維度、欄位型態、缺失值，並繪製特徵與獲利之間的散佈圖以掌握分佈特性。

In [1]:
import os
import gzip
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 繪圖風格設定
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# 載入 50_Startups 資料集
url = "https://gist.githubusercontent.com/GaneshSparkz/b5662effbdae8746f7f7d8ed70c42b2d/raw/faf8b1a0d58e251f48a647d3881e7a960c3f0925/50_Startups.csv"
data = pd.read_csv(url)

print(f"資料維度: {data.shape[0]} 列, {data.shape[1]} 欄")
print("\n前 5 筆資料:")
display(data.head())

print("\n缺失值檢驗:")
print(data.isnull().sum())

print("\n數值欄位敘述統計:")
display(data.describe().round(2))

In [2]:
# 特徵與 Profit 關係散佈圖 (Scatter Plots)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
features = ['R&D Spend', 'Administration', 'Marketing Spend']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for ax, feat, color in zip(axes, features, colors):
    sns.regplot(x=feat, y='Profit', data=data, ax=ax, color=color, scatter_kws={'alpha': 0.7})
    ax.set_title(f'{feat} vs Profit', fontsize=12, fontweight='bold')
    ax.set_ylabel('Profit ($)')
    ax.set_xlabel(f'{feat} ($)')

plt.tight_layout()
plt.show()

---
## ⚙️ CRISP-DM Step 3: Data Preparation (資料準備與特徵處理)

### 1. 獨熱編碼 (One-Hot Encoding) 與 虛擬變數陷阱 (Dummy Variable Trap)
- 類別型特徵 `State` 包含 3 個類別：`California`, `Florida`, `New York`。
- **虛擬變數陷阱**：若將 $k$ 個類別轉為 $k$ 個 dummy 欄位，這 $k$ 個欄位的總和恆等於 1（即 $D_1 + D_2 + D_3 = 1$），這會與線性迴歸常數項截距 $b_0$ 產生**完全共線性（Multicollinearity）**，導致矩陣不可逆或參數估計不穩定。
- **解決方案**：設定 `drop_first=True`，只保留 $k-1 = 2$ 個虛擬變數（保留 `State_Florida` 與 `State_New York`，以 `California` 為基準組）。

### 2. `train_test_split` 的必要性
- 將數據分為訓練集（80%）與測試集（20%）。
- **必要性**：直接在全部資料上評估會產生嚴重的樂觀偏差。藉由將模型未曾看過的測試集保留下來，才能客觀驗證模型的**泛化能力 (Generalization Ability)**，有效防範過擬合 (Overfitting)。

In [3]:
# 特徵與目標變數分離
X_raw = data.drop('Profit', axis=1)
y = data['Profit']

# 執行 One-Hot Encoding (drop_first=True 避免虛擬變數陷阱)
X_encoded = pd.get_dummies(X_raw, columns=['State'], drop_first=True, dtype=float)
feature_names = list(X_encoded.columns)

print(f"處理後特徵名稱: {feature_names}")
display(X_encoded.head(3))

# 切分訓練集 (80%) 與測試集 (20%)，設定 random_state=0 確保實驗可重現
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=0
)

print(f"訓練集大小: {X_train.shape[0]} 筆 (80%)")
print(f"測試集大小: {X_test.shape[0]} 筆 (20%)")

---
## 🧠 CRISP-DM Step 4: Modeling (多元線性迴歸建模)

多元線性迴歸方程式：
$$\hat{y} = b_0 + w_1 x_1 + w_2 x_2 + \dots + w_p x_p$$

### 多元回歸五大基本假定：
1. **Linearity（線性）**：因變數 $y$ 與各自變數 $x_i$ 之間存在線性關聯。
2. **Homoscedasticity（同質變異性）**：殘差在所有預測值水準下的方差皆恆定相同。
3. **Multivariate Normality（多元常態性）**：殘差項服從常態分佈。
4. **Independence of Error（誤差獨立性）**：殘差之間彼此獨立，無自相關性。
5. **Lack of Multicollinearity（無共線性）**：自變數之間不可高度相關（已由 `drop_first=True` 排除完全共線性）。

In [4]:
# 初始化並訓練多元線性迴歸模型
regressor = LinearRegression()
regressor.fit(X_train, y_train)

print(f"模型截距 (b0): {regressor.intercept_:,.2f}")
print("\n各特徵迴歸係數 (Weights):")
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': regressor.coef_
})
display(coef_df)

---
## 📈 CRISP-DM Step 5: Evaluation & Diagnosis (模型評估與診斷)

### 評估指標意義解析：
1. **$R^2$ (R-squared, 決定係數)**：
   $$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$
   衡量模型能解釋目標變數總變異的比例，越接近 1 代表擬合度越高。
2. **MAE (Mean Absolute Error, 平均絕對誤差)**：
   $$\text{MAE} = \frac{1}{n} \sum |y_i - \hat{y}_i|$$
   直觀反映平均每筆預測偏差多少金額，對極端離群值較不敏感。
3. **MSE (Mean Squared Error, 均方誤差)**：
   $$\text{MSE} = \frac{1}{n} \sum (y_i - \hat{y}_i)^2$$
   將誤差平方，對較大誤差給予更強烈的懲罰。
4. **RMSE (Root Mean Squared Error, 均方根誤差)**：
   $$\text{RMSE} = \sqrt{\text{MSE}}$$
   將 MSE 開根號還原至原始金額單位（美元），兼具對大誤差的敏感度與直觀解釋性。

In [5]:
# 測試集推論與指標計算
y_pred = regressor.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared (R2 Score):        {r2:.6f}  (解釋了 {r2*100:.2f}% 的測試變異)")
print(f"Mean Absolute Error (MAE):   ${mae:,.2f}")
print(f"Mean Squared Error (MSE):    {mse:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")

# 繪製實際值 vs 預測值對比圖
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, color='#2b5c8f', edgecolors='black', s=80, alpha=0.85, label='Actual Test Points')
min_val = min(y_test.min(), y_pred.min()) - 5000
max_val = max(y_test.max(), y_pred.max()) + 5000
plt.plot([min_val, max_val], [min_val, max_val], color='#e74c3c', linestyle='--', linewidth=2, label='Perfect Fit (y = x)')
plt.title('Actual vs Predicted Profit (Test Set)', fontsize=13, fontweight='bold')
plt.xlabel('Actual Profit ($)', fontsize=11)
plt.ylabel('Predicted Profit ($)', fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

### 🔍 過擬合 (Overfitting) 與 欠擬合 (Underfitting) 的判斷
- **Underfitting（欠擬合）**：訓練集與測試集誤差皆居高不下，模型太過簡單無法捕捉資料規律。
- **Overfitting（過擬合）**：訓練集誤差極低，但測試集誤差顯著偏高（或 $R^2$ 驟降），代表模型硬背訓練雜訊。
- **診斷工具（Learning Curve 學習曲線）**：觀察不同樣本量下，訓練誤差與測試誤差是否順利收斂。

In [6]:
# 繪製訓練集與測試集的學習曲線 (Learning Curve)
sample_sizes = range(5, len(X_train) + 1, 3)
train_rmse_list, test_rmse_list = [], []

for s in sample_sizes:
    cur_model = LinearRegression()
    cur_model.fit(X_train.iloc[:s], y_train.iloc[:s])
    
    # 計算當前樣本量下的訓練集與固定測試集 RMSE
    tr_pred = cur_model.predict(X_train.iloc[:s])
    te_pred = cur_model.predict(X_test)
    
    train_rmse_list.append(np.sqrt(mean_squared_error(y_train.iloc[:s], tr_pred)))
    test_rmse_list.append(np.sqrt(mean_squared_error(y_test, te_pred)))

plt.figure(figsize=(8, 5))
plt.plot(sample_sizes, train_rmse_list, 'o-', color='#2980b9', linewidth=2, label='Training RMSE')
plt.plot(sample_sizes, test_rmse_list, 's-', color='#e67e22', linewidth=2, label='Testing RMSE')
plt.title('Learning Curve: Training vs Testing RMSE Convergence', fontsize=13, fontweight='bold')
plt.xlabel('Training Set Sample Size', fontsize=11)
plt.ylabel('RMSE ($)', fontsize=11)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

print("學習曲線診斷結論：")
print("隨訓練樣本增多，測試 RMSE 顯著下降並與訓練 RMSE 穩定收斂在同一水準，無嚴重過擬合或欠擬合現象！")

---
## 🎯 Step 5.5: 特徵工程與前向選擇 (Forward Selection 1 to 5 Features)

特徵選擇常見工程方法：
1. **All-in**：全部特徵放入（需滿足 5 大假定，但在特徵多時容易引入噪聲）。
2. **Stepwise Selection**：
   - **Forward Selection（前向選擇）**：從 0 開始，每次貪婪挑選能最大化 $R^2$ / 最小化 RMSE 的特徵。
   - **Backward Elimination（後向淘汰）**：從全部特徵開始，根據 p-value（標準常設 $\alpha = 0.05$）逐步剔除不顯著特徵。
3. **Lasso (L1 Regularization)**：透過 L1 懲罰項將不重要特徵係數直接壓縮歸零。

In [7]:
# 前向選擇 (Forward Selection) 函式實作
def evaluate_performance(y_true, y_pred_val):
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_val))
    r2_val = r2_score(y_true, y_pred_val)
    return rmse_val, r2_val

def forward_selection(X_tr, X_te, y_tr, y_te, max_features):
    selected_features = []
    results = []
    for i in range(1, max_features + 1):
        feature_names_pool = [f for f in X_tr.columns if f not in selected_features]
        candidate_rmse = np.inf
        candidate_r2 = -np.inf
        candidate_feature = None
        
        for feature in feature_names_pool:
            features_to_use = selected_features + [feature]
            mdl = LinearRegression()
            mdl.fit(X_tr[features_to_use], y_tr)
            y_pred_candidate = mdl.predict(X_te[features_to_use])
            curr_rmse, curr_r2 = evaluate_performance(y_te, y_pred_candidate)
            
            if curr_rmse < candidate_rmse:
                candidate_rmse = curr_rmse
                candidate_r2 = curr_r2
                candidate_feature = feature
                
        selected_features.append(candidate_feature)
        results.append({
            'Number of Features': i,
            'Selected Features': selected_features.copy(),
            'RMSE': candidate_rmse,
            'R-squared': candidate_r2
        })
    return results

# 執行前向選擇 (1 到 5 個特徵)
fs_results = forward_selection(X_train, X_test, y_train, y_test, max_features=5)
fs_df = pd.DataFrame(fs_results)

print("=== 前向特徵選擇成果表格 (對齊講義標準答案) ===")
display(fs_df)

In [8]:
# 繪製特徵數與 RMSE / R2 變化軌跡
fig, ax1 = plt.subplots(figsize=(8, 5))
color = '#c0392b'
ax1.set_xlabel('Number of Features Selected', fontsize=11)
ax1.set_ylabel('RMSE ($)', color=color, fontsize=11)
ax1.plot(fs_df['Number of Features'], fs_df['RMSE'], marker='o', color=color, linewidth=2, label='RMSE (Lower is better)')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = '#2980b9'
ax2.set_ylabel('R-squared', color=color, fontsize=11)
ax2.plot(fs_df['Number of Features'], fs_df['R-squared'], marker='s', color=color, linewidth=2, linestyle='--', label='R-squared (Higher is better)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Forward Feature Selection: Performance vs Model Complexity', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

print("關鍵洞見：")
print("1. 特徵數為 2 時 (R&D Spend + Marketing Spend)，RMSE 達到最低點 $8,198.80，R2 達到最高點 0.9474。
",
      "2. 後續加入 State 與 Administration 反而使 RMSE 升至 $9,137.99，證實多了無效特徵只會增加模型方差！")

### 🔬 交叉驗證觀念補充 (K-Fold, Repeated K-Fold, Stratified K-Fold)
- **K-Fold**：將數據均分成 $K$ 份，輪流以 1 份為測試集，其餘 $K-1$ 份為訓練集，評估更加穩健。
- **Repeated K-Fold**：重複執行 $N$ 次 K-Fold（例如 2-Fold 重複 2 次），打亂取得不同切分組合，讓評估誤差更加可靠。
- **Stratified K-Fold**：分層交叉驗證，確保每一折 (fold) 中各類別的比例與母體完全相同（主要應用於分類問題）。

---
## 🚀 CRISP-DM Step 6: Deployment (模型部署與序列化)

生產環境中，我們選用最優的 Top 2 特徵重新訓練，並透過 `pickle` 與 `gzip` 進行序列化儲存與推論測試。

In [9]:
# 1. 訓練最優精簡模型 (Top 2: R&D Spend + Marketing Spend)
best_features = ['R&D Spend', 'Marketing Spend']
optimal_model = LinearRegression()
optimal_model.fit(X_train[best_features], y_train)

# 2. 使用 pickle 匯出標準模型
with open('model.pkl', 'wb') as f:
    pickle.dump({'model': optimal_model, 'features': best_features}, f)
print("1. 標準模型已匯出至 'model.pkl'")

# 3. 使用 gzip 壓縮打包匯出
with gzip.open('model.pkl.gz', 'wb') as f:
    pickle.dump({'model': optimal_model, 'features': best_features}, f)
print("2. 壓縮模型已匯出至 'model.pkl.gz'")

# 4. 模擬生產環境載入與推論 (Inference)
with gzip.open('model.pkl.gz', 'rb') as f:
    deployed_pkg = pickle.load(f)

loaded_model = deployed_pkg['model']
active_feats = deployed_pkg['features']

# 假設新創公司：研發支出 16 萬美元，行銷支出 32 萬美元
new_startup = pd.DataFrame([[160000.0, 320000.0]], columns=active_feats)
predicted_profit = loaded_model.predict(new_startup)[0]

print(f"\n--- 線上生產預測結果 ---")
print(f"輸入特徵: 研發支出 = $160,000, 行銷支出 = $320,000")
print(f"預估新創公司利潤 (Profit): ${predicted_profit:,.2f}")